# End-to-End EDA on Zomato Restaurant Reviews

## Loading Data and Initial Exploration

In [7]:
# Import Libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

In [8]:
# Load datasets
reviews_df = pd.read_csv('Zomato Restaurant reviews.csv')
metadata_df = pd.read_csv('Zomato Restaurant names and Metadata.csv')

print(f"reviews_df  -> {reviews_df.shape[0]} rows, {reviews_df.shape[1]} columns")
print(f"metadata_df -> {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns")

reviews_df  -> 10000 rows, 7 columns
metadata_df -> 105 rows, 6 columns


In [9]:
# Display first 5 rows of both datasets
reviews_df.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0


In [10]:
metadata_df.head()

,Name,Links,Cost,Collections,Cuisines,Timings
0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
1,Paradise,https://www.zomato.com/hyderabad/paradise-gach...,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese",11 AM to 11 PM
2,Flechazo,https://www.zomato.com/hyderabad/flechazo-gach...,"1,300","Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts","11:30 AM to 4:30 PM, 6:30 PM to 11 PM"
3,Shah Ghouse Hotel & Restaurant,https://www.zomato.com/hyderabad/shah-ghouse-h...,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever...",12 Noon to 2 AM
4,Over The Moon Brew Company,https://www.zomato.com/hyderabad/over-the-moon...,"1,200","Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med...","12noon to 11pm (Mon, Tue, Wed, Thu, Sun), 12no..."


In [11]:
# Data types, columns, shape
print("=== reviews_df.info() ===")
reviews_df.info()
print("\n=== metadata_df.info() ===")
metadata_df.info()

=== reviews_df.info() ===
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Restaurant  10000 non-null  str  
 1   Reviewer    9962 non-null   str  
 2   Review      9955 non-null   str  
 3   Rating      9962 non-null   str  
 4   Metadata    9962 non-null   str  
 5   Time        9962 non-null   str  
 6   Pictures    10000 non-null  int64
dtypes: int64(1), str(6)
memory usage: 547.0 KB

=== metadata_df.info() ===
<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Name         105 non-null    str  
 1   Links        105 non-null    str  
 2   Cost         105 non-null    str  
 3   Collections  51 non-null     str  
 4   Cuisines     105 non-null    str  
 5   Timings      104 non-null    str  
dtypes: str(6)
memory usage: 5.1 KB


## Data Cleaning

### Missing Values

In [12]:
# Check for missing values
print("=== Missing values: reviews_df ===")
print(reviews_df.isnull().sum())
print("\n=== Missing values: metadata_df ===")
print(metadata_df.isnull().sum())

=== Missing values: reviews_df ===
Restaurant     0
Reviewer      38
Review        45
Rating        38
Metadata      38
Time          38
Pictures       0
dtype: int64

=== Missing values: metadata_df ===
Name            0
Links           0
Cost            0
Collections    54
Cuisines        0
Timings         1
dtype: int64


In [13]:
# Handle missing values
# 38 rows in reviews_df are missing Reviewer/Review/Rating/Metadata/Time together -> these are
# fully blank review rows (no usable content), so they are dropped rather than imputed
blank_review_rows = reviews_df['Rating'].isnull().sum()
print(f"Fully blank review rows to drop: {blank_review_rows}")
reviews_df = reviews_df.dropna(subset=['Rating', 'Review']).reset_index(drop=True)

# metadata_df: Collections is an optional tag field (54/105 missing) -> fill with 'None Listed'
metadata_df['Collections'] = metadata_df['Collections'].fillna('None Listed')
# Timings: 1 missing -> fill with 'Not Available'
metadata_df['Timings'] = metadata_df['Timings'].fillna('Not Available')

print("\nRemaining nulls, reviews_df:\n", reviews_df.isnull().sum().sum())
print("Remaining nulls, metadata_df:\n", metadata_df.isnull().sum().sum())

Fully blank review rows to drop: 38

Remaining nulls, reviews_df:
 0
Remaining nulls, metadata_df:
 0


### Removing Duplicates

In [14]:
# Check for and remove duplicate rows
print(f"Duplicate rows in reviews_df: {reviews_df.duplicated().sum()}")
print(f"Duplicate rows in metadata_df: {metadata_df.duplicated().sum()}")

reviews_df.drop_duplicates(inplace=True)
metadata_df.drop_duplicates(inplace=True)
reviews_df.reset_index(drop=True, inplace=True)

print(f"\nShape after dedup -> reviews_df: {reviews_df.shape}, metadata_df: {metadata_df.shape}")

Duplicate rows in reviews_df: 0
Duplicate rows in metadata_df: 0

Shape after dedup -> reviews_df: (9955, 7), metadata_df: (105, 6)


### Correcting Data Types 

In [15]:
# Rating contains a non-numeric value ('Like') alongside numeric strings -> coerce to numeric,
# turning 'Like' into NaN, then drop those (a handful of rows with no real star rating)
reviews_df['Rating'] = pd.to_numeric(reviews_df['Rating'], errors='coerce')
print(f"Rows with non-numeric Rating ('Like', etc.): {reviews_df['Rating'].isnull().sum()}")
reviews_df = reviews_df.dropna(subset=['Rating']).reset_index(drop=True)

# Cost has thousands-separator commas as text (e.g. '1,300') -> strip commas, cast to numeric
metadata_df['Cost'] = metadata_df['Cost'].str.replace(',', '', regex=False).astype(float)

# Time -> proper datetime
reviews_df['Time'] = pd.to_datetime(reviews_df['Time'], format='%m/%d/%Y %H:%M')

# Parse the 'Metadata' text field ('3 Reviews , 2 Followers') into numeric reviewer stats
def extract_count(pattern, text):
    match = re.search(pattern, str(text))
    return int(match.group(1)) if match else 0

reviews_df['Reviewer_Review_Count'] = reviews_df['Metadata'].apply(
    lambda x: extract_count(r'(\d+)\s+Reviews?', x))
reviews_df['Reviewer_Follower_Count'] = reviews_df['Metadata'].apply(
    lambda x: extract_count(r'(\d+)\s+Followers?', x))

print(reviews_df.dtypes)
print()
print(metadata_df.dtypes)

Rows with non-numeric Rating ('Like', etc.): 1
Restaurant                            str
Reviewer                              str
Review                                str
Rating                            float64
Metadata                              str
Time                       datetime64[us]
Pictures                            int64
Reviewer_Review_Count               int64
Reviewer_Follower_Count             int64
dtype: object

Name               str
Links              str
Cost           float64
Collections        str
Cuisines           str
Timings            str
dtype: object


In [16]:
# Merge the two datasets on restaurant name for the analyses that need both (e.g. Cost vs Rating)
merged_df = reviews_df.merge(metadata_df, left_on='Restaurant', right_on='Name', how='left')
print(f"Merged shape: {merged_df.shape}")
print(f"Unmatched rows after merge: {merged_df['Name'].isnull().sum()}")
merged_df.head(3)

Merged shape: (9954, 15)
Unmatched rows after merge: 0


,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures,Reviewer_Review_Count,Reviewer_Follower_Count,Name,Links,Cost,Collections,Cuisines,Timings
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5.0,"1 Review , 2 Followers",2019-05-25 15:54:00,0,1,2,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800.0,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5.0,"3 Reviews , 2 Followers",2019-05-25 14:20:00,0,3,2,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800.0,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5.0,"2 Reviews , 3 Followers",2019-05-24 22:54:00,0,2,3,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800.0,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
